# Bonus — Evaluación y Producción: Práctica

**Objetivo:** Implementar las herramientas que diferencian un sistema de IA en producción de un prototipo: tracing manual, golden datasets, tests de regresión y load testing asíncrono.

Un sistema de IA sin observabilidad y evaluación continua es un sistema que no puedes operar con confianza. Estas prácticas construyen la infraestructura mínima viable para operar en producción.

**Prerequisito:** Variable de entorno `OPENAI_API_KEY` configurada.

## Setup

In [ ]:
!pip install openai --quiet
import time
import statistics
print("Módulos listos: openai, time, statistics")

## Ejercicio 1 — Tracing manual con decorador

**Qué practicamos:** Implementar un decorador que registra automáticamente la entrada, salida, duración y tokens de cada llamada LLM, generando un trace estructurado.

**Por qué importa arquitectónicamente:** Sin tracing, cuando algo falla en producción no sabes qué input causó el fallo, qué output se generó, ni cuánto tardó. El trace es el registro de auditoría del sistema. Herramientas como LangSmith o Arize hacen esto, pero entender la implementación mínima te permite construir el tuyo propio o evaluar si necesitas una herramienta externa.

In [ ]:
import os
import time
import json
import functools
import uuid
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Almacenamiento de traces en memoria
# ponytail: lista en memoria. En producción: escribir a log estructurado (JSON lines) o enviar a observability platform.
TRACES: list[dict] = []

def trace_llm(fn):
    """Decorador que añade tracing automático a cualquier función que llame al LLM."""
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        trace_id = str(uuid.uuid4())[:8]
        start = time.time()
        error = None
        result = None
        usage = None

        try:
            result = fn(*args, **kwargs)
            # Si la función devuelve una tupla (output, usage_dict), extraer uso
            if isinstance(result, tuple) and len(result) == 2 and isinstance(result[1], dict):
                result, usage = result
        except Exception as e:
            error = str(e)
            raise
        finally:
            duration_ms = int((time.time() - start) * 1000)
            trace = {
                "id":          trace_id,
                "function":    fn.__name__,
                "args":        str(args)[:200],
                "kwargs":      str(kwargs)[:200],
                "duration_ms": duration_ms,
                "result":      str(result)[:300] if result else None,
                "usage":       usage,
                "error":       error,
            }
            TRACES.append(trace)
            status = "ERROR" if error else "OK"
            print(f"[TRACE {trace_id}] {fn.__name__} | {duration_ms}ms | {status}")
            if usage:
                print(f"            tokens: {usage}")

        return result
    return wrapper

@trace_llm
def classify_intent(text: str) -> tuple[str, dict]:
    """Clasifica la intención del texto. Devuelve (intent, usage)."""
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Clasifica la intención en una palabra: question, complaint, request, compliment."},
            {"role": "user",   "content": text}
        ],
        temperature=0, max_tokens=10
    )
    usage = {"prompt": r.usage.prompt_tokens, "completion": r.usage.completion_tokens}
    return r.choices[0].message.content.strip(), usage

@trace_llm
def generate_reply(intent: str, original_text: str) -> str:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Responde a un mensaje con intención '{intent}' de forma profesional. Máximo 2 frases."},
            {"role": "user",   "content": original_text}
        ],
        temperature=0.3, max_tokens=80
    )
    return r.choices[0].message.content

# Pipeline trazado
messages = [
    "¿Cuánto cuesta el plan premium?",
    "El servicio de soporte es un desastre total",
    "Me encanta la nueva interfaz, muy bien diseñada",
]

for msg in messages:
    intent = classify_intent(msg)
    reply  = generate_reply(intent, msg)
    print(f"MSG:    '{msg}'")
    print(f"INTENT: {intent}")
    print(f"REPLY:  {reply[:100]}")
    print()

print(f"\nTotal de traces almacenados: {len(TRACES)}")
print("Durations:", [t['duration_ms'] for t in TRACES])

## Ejercicio 2 — Golden Dataset: evaluar el pipeline contra casos conocidos

**Qué practicamos:** Crear un dataset de inputs con outputs esperados y una función que evalúa el pipeline contra ellos, calculando un score de calidad.

**Por qué importa arquitectónicamente:** Un golden dataset es el contrato de comportamiento del sistema. Cada vez que cambias el modelo, el prompt o la arquitectura, corres el dataset y ves si el score sube o baja. Sin esto, cualquier cambio en producción es un salto de fe. Es el equivalente a los tests de integración para sistemas de IA.

In [ ]:
import os
import json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# El pipeline a evaluar
def pipeline(question: str) -> str:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Eres un asistente experto en arquitectura de sistemas de IA. Responde de forma precisa y concisa."},
            {"role": "user",   "content": question}
        ],
        temperature=0, max_tokens=150
    )
    return r.choices[0].message.content

# LLM judge para comparar semánticamente
def semantic_score(question: str, expected: str, actual: str) -> float:
    prompt = f"""¿La respuesta real cubre el mismo concepto que la esperada?
Pregunta: {question}
Esperada: {expected}
Real: {actual}
Responde con JSON: {{"score": 0.0-1.0, "note": "una frase"}}"""

    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0
    )
    return json.loads(r.choices[0].message.content)

# Golden dataset
GOLDEN = [
    {
        "id": "g001",
        "question": "¿Cuándo usar RAG en vez de fine-tuning?",
        "expected": "RAG cuando el conocimiento cambia frecuentemente o es específico del dominio. Fine-tuning cuando necesitas cambiar el comportamiento o el estilo del modelo."
    },
    {
        "id": "g002",
        "question": "¿Qué es la temperatura en un LLM y cuándo usar temperature=0?",
        "expected": "La temperatura controla la aleatoriedad del output. Temperature=0 para outputs deterministas y reproducibles: pipelines, tests, extracción de datos."
    },
    {
        "id": "g003",
        "question": "¿Cuál es la diferencia entre un agente y un pipeline de IA?",
        "expected": "Un pipeline tiene flujo fijo predefinido. Un agente decide dinámicamente qué pasos seguir según el contexto, usando el LLM como motor de razonamiento."
    },
]

def evaluate_pipeline(golden: list[dict], threshold: float = 0.7) -> dict:
    results = []
    for case in golden:
        actual = pipeline(case["question"])
        evaluation = semantic_score(case["question"], case["expected"], actual)
        results.append({
            "id":       case["id"],
            "score":    evaluation["score"],
            "passed":   evaluation["score"] >= threshold,
            "note":     evaluation["note"],
            "actual":   actual[:100]
        })
        status = "PASS" if evaluation["score"] >= threshold else "FAIL"
        print(f"[{status}] {case['id']}: score={evaluation['score']:.2f} | {evaluation['note']}")

    avg_score = sum(r["score"] for r in results) / len(results)
    passed = sum(1 for r in results if r["passed"])
    return {"avg_score": avg_score, "passed": passed, "total": len(results), "results": results}

print(f"Evaluando pipeline contra {len(GOLDEN)} casos golden...\n")
report = evaluate_pipeline(GOLDEN)
print(f"\n=== RESULTADO ===")
print(f"Score promedio: {report['avg_score']:.2f}")
print(f"Casos pasados:  {report['passed']}/{report['total']}")

## Ejercicio 3 — Test de regresión que falla si el score baja del threshold

**Qué practicamos:** Convertir la evaluación del ejercicio anterior en un test automatizado que falla explícitamente si el score baja del umbral mínimo aceptable.

**Por qué importa arquitectónicamente:** Los tests de regresión para IA son los guardabarreras del CI/CD. Cuando un cambio de modelo, prompt o arquitectura degrada la calidad, el test lo detecta antes de llegar a producción. Sin esta red de seguridad, los cambios se despliegan a ciegas.

In [ ]:
import os
import json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MIN_AVG_SCORE = 0.65  # umbral mínimo de calidad del pipeline
MIN_PASS_RATE = 0.6   # al menos el 60% de los casos deben pasar

def quick_pipeline(question: str) -> str:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Responde de forma precisa y concisa sobre IA y arquitectura de sistemas."},
            {"role": "user", "content": question}
        ],
        temperature=0, max_tokens=100
    )
    return r.choices[0].message.content

def judge_score(question: str, expected: str, actual: str) -> float:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"""Puntúa la cobertura conceptual de 0.0 a 1.0.
Q: {question}
Esperado: {expected}
Real: {actual}
JSON: {{"score": 0.0-1.0}}"""}],
        response_format={"type": "json_object"},
        temperature=0
    )
    return json.loads(r.choices[0].message.content)["score"]

GOLDEN_MINI = [
    {"q": "¿Qué es un token?",
     "a": "Unidad mínima de procesamiento de un LLM. Aproximadamente 0.75 palabras en inglés."},
    {"q": "¿Para qué sirve la ventana de contexto?",
     "a": "Determina cuánta información puede procesar el LLM en una sola inferencia. Es el límite de memoria de trabajo del modelo."},
    {"q": "¿Qué problema resuelve el chunking en RAG?",
     "a": "Divide documentos en fragmentos del tamaño correcto para que el retrieval encuentre información precisa sin incluir ruido innecesario."},
]

def regression_test():
    """Test que falla si el pipeline baja del threshold de calidad."""
    print("Ejecutando test de regresión...\n")
    scores = []
    passed = 0

    for case in GOLDEN_MINI:
        actual = quick_pipeline(case["q"])
        score = judge_score(case["q"], case["a"], actual)
        scores.append(score)
        ok = score >= 0.6
        if ok:
            passed += 1
        print(f"  {'PASS' if ok else 'FAIL'} [{score:.2f}] {case['q'][:60]}")

    avg = sum(scores) / len(scores)
    pass_rate = passed / len(scores)

    print(f"\nScore promedio: {avg:.2f} (mínimo: {MIN_AVG_SCORE})")
    print(f"Pass rate:      {pass_rate:.0%} (mínimo: {MIN_PASS_RATE:.0%})")

    # Assertions que fallan el test si la calidad degrada
    assert avg >= MIN_AVG_SCORE, f"REGRESION: score promedio {avg:.2f} < mínimo {MIN_AVG_SCORE}"
    assert pass_rate >= MIN_PASS_RATE, f"REGRESION: pass rate {pass_rate:.0%} < mínimo {MIN_PASS_RATE:.0%}"

    print("\n✓ Test de regresión PASSED")
    return {"avg": avg, "pass_rate": pass_rate}

try:
    result = regression_test()
except AssertionError as e:
    print(f"\n✗ Test de regresión FAILED: {e}")

## Ejercicio 4 — Load Test asíncrono: p50/p95/p99 de latencia

**Qué practicamos:** Enviar 50 requests en paralelo usando `asyncio` y calcular los percentiles de latencia p50, p95 y p99.

**Por qué importa arquitectónicamente:** La latencia media es engañosa. El p95 y p99 revelan los casos extremos que afectan al 5% y 1% de los usuarios — que en producción de alto volumen son miles de usuarios. Un sistema con p50=1s y p99=15s tiene un problema serio aunque la media sea 1.5s. Los percentiles son la métrica correcta para SLAs.

In [ ]:
import os
import asyncio
import time
import statistics
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

N_REQUESTS = 50
MAX_CONCURRENT = 10  # rate limiting básico

async def single_request(idx: int, semaphore: asyncio.Semaphore) -> dict:
    """Un request individual con medición de latencia."""
    async with semaphore:
        start = time.time()
        error = None
        try:
            r = await async_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": f"Responde en una sola palabra: ¿qué es un LLM? (request {idx})"}],
                temperature=0, max_tokens=5
            )
            output = r.choices[0].message.content.strip()
        except Exception as e:
            error = str(e)
            output = None
        latency_ms = int((time.time() - start) * 1000)
        return {"idx": idx, "latency_ms": latency_ms, "output": output, "error": error}

async def load_test(n: int, max_concurrent: int) -> list[dict]:
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [single_request(i, semaphore) for i in range(n)]
    return await asyncio.gather(*tasks)

def percentile(data: list[float], p: int) -> float:
    sorted_data = sorted(data)
    idx = max(0, int(len(sorted_data) * p / 100) - 1)
    return sorted_data[idx]

print(f"Ejecutando load test: {N_REQUESTS} requests, max {MAX_CONCURRENT} concurrentes...\n")
wall_start = time.time()
results = asyncio.run(load_test(N_REQUESTS, MAX_CONCURRENT))
wall_time = time.time() - wall_start

# Análisis de resultados
latencies = [r["latency_ms"] for r in results if r["error"] is None]
errors = [r for r in results if r["error"] is not None]

print(f"Completados: {len(latencies)}/{N_REQUESTS} exitosos | {len(errors)} errores")
print(f"Tiempo total (wall clock): {wall_time:.2f}s")
print(f"Throughput: {len(latencies)/wall_time:.1f} req/s")
print()
print("=== Latencia (ms) ===")
print(f"  min:  {min(latencies)}")
print(f"  p50:  {percentile(latencies, 50)}")
print(f"  p95:  {percentile(latencies, 95)}")
print(f"  p99:  {percentile(latencies, 99)}")
print(f"  max:  {max(latencies)}")
print(f"  mean: {statistics.mean(latencies):.0f}")
print(f"  stdev:{statistics.stdev(latencies):.0f}")

if errors:
    print(f"\n=== Errores ===")
    for e in errors[:3]:
        print(f"  Request {e['idx']}: {e['error']}")